In [3]:
import os
import json
import cv2
import numpy as np
from tqdm import tqdm

In [7]:
# Paths
root_dir = r"C:\Users\patel\Downloads\Lane_detection\bdd100k"
image_dir = os.path.join(root_dir, "images", "100k", "train")
label_dir = os.path.join(root_dir, "labels", "100k", "train")
output_mask_dir = os.path.join(root_dir, "processed", "masks", "train")
output_img_dir = os.path.join(root_dir, "processed", "images", "train")

In [8]:

os.makedirs(output_mask_dir, exist_ok=True)
os.makedirs(output_img_dir, exist_ok=True)


In [9]:
# Process each label
for file in tqdm(os.listdir(label_dir)):
    if not file.endswith(".json"):
        continue

    label_path = os.path.join(label_dir, file)
    with open(label_path, "r") as f:
        data = json.load(f)

    img_name = data["name"] + ".jpg"  # or check actual extension
    img_path = os.path.join(image_dir, img_name)
    if not os.path.exists(img_path):
        continue

    # Load image to get shape
    img = cv2.imread(img_path)
    if img is None:
        continue

    h, w = img.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    # Loop through annotations
    for frame in data["frames"]:
        for obj in frame["objects"]:
            cat = obj["category"]
            if "poly2d" in obj and cat.startswith("lane/"):
                pts = np.array([[p[0], p[1]] for p in obj["poly2d"]], np.int32)
                pts = pts.reshape((-1, 1, 2))
                cv2.polylines(mask, [pts], isClosed=False, color=255, thickness=5)
    # Resize for training (optional)
    mask_resized = cv2.resize(mask, (512, 256))
    img_resized = cv2.resize(img, (512, 256))
    # Save results
    cv2.imwrite(os.path.join(output_mask_dir, file.replace(".json", ".png")), mask_resized)
    cv2.imwrite(os.path.join(output_img_dir, img_name), img_resized)

100%|████████████████████████████████████████████████████████████████████████████| 70000/70000 [20:46<00:00, 56.15it/s]


In [10]:
print("✅ Preprocessing complete. Lane masks saved in:", output_mask_dir)

✅ Preprocessing complete. Lane masks saved in: C:\Users\patel\Downloads\Lane_detection\bdd100k\processed\masks\train
